# NORIA — Stage 2 (LivePortrait, Python 3.11 fix)

**Root cause of earlier failures:** Colab now runs **Python 3.13**, but these
face models need **Python 3.11** — their libraries (numpy 1.26, scipy 1.13,
onnxruntime 1.18) have no 3.13 builds, so pip tried to compile them and broke.

**Fix:** we build a real **Python 3.11 environment** with `uv` and run everything
inside it, where every library installs as a prebuilt wheel — fast and reliable.

**Run:** set GPU (`Runtime → Change runtime type → T4 GPU`), then `Runtime → Run
all`. ~6–9 min. Result (a photoreal animated Noria) appears in the last cell.

## 0. GPU on?

In [ ]:
!nvidia-smi -L

## 1. Build a Python 3.11 environment (the real fix)

In [ ]:
!pip -q install uv
!uv python install 3.11
%cd /content
!git clone -q https://github.com/KwaiVGI/LivePortrait || echo "already cloned"
%cd /content/LivePortrait
!uv venv --python 3.11 .venv
print('py3.11 env created')

## 2. Install LivePortrait into the 3.11 env (prebuilt wheels — no compiling)

In [ ]:
# GPU torch for py3.11 (CUDA 12 wheels work with Colab's driver)
!uv pip install --python .venv/bin/python torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121
# the rest of LivePortrait's deps (all have 3.11 wheels)
!uv pip install --python .venv/bin/python -r requirements.txt huggingface_hub
!.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 3. Download the model weights

In [ ]:
!.venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('KwaiVGI/LivePortrait', local_dir='pretrained_weights', allow_patterns=['*.pth','*.onnx','*.safetensors','*.json','*.txt']); print('weights ready')"

## 4. Get Noria's face
Pulls her real render from your live site (`noria-m.png` for Noria-M).

In [ ]:
!wget -q -O /content/noria.png https://noria-body.onrender.com/assets/noria-f.png
from IPython.display import Image
Image('/content/noria.png', width=260)

## 5. Animate Noria (built-in reference motion clip)

In [ ]:
import glob, os
os.chdir('/content/LivePortrait')
driver = sorted(glob.glob('assets/examples/driving/*.mp4'))[0]
print('driver clip:', os.path.basename(driver))
!.venv/bin/python inference.py -s /content/noria.png -d {driver}

## 6. Watch her — and download

In [ ]:
import glob, os
from IPython.display import HTML
from base64 import b64encode
outs = sorted(glob.glob('/content/LivePortrait/animations/*.mp4'), key=os.path.getmtime)
assert outs, 'No video produced — check the previous cell output.'
concat = [o for o in outs if 'concat' in o]
mp4 = (concat or outs)[-1]
print('Animated Noria:', mp4)
data = b64encode(open(mp4,'rb').read()).decode()
display(HTML(f'<video width=400 controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))
try:
    from google.colab import files; files.download(mp4)
except Exception:
    print('Download from the Files panel:', mp4)